# CARE Score and Care2CompareDataset usage

To use the CARE-Score with other datasets you need the following data:
- define events containing anomalous data (the period before an actual fault)
- define events containing normal data
- For all events there needs to be enough training data beforehand to train an early fault detection model
- Ideally you have the same amount of normal and anomalous events

Then you can use the `CAREScore` class to calculate the final score:
- Create models and get predictions for each event.
- Evaluate each event using the `CAREScore.evaluate_event` method
- Calculate the CARE score `CAREScore.get_final_score`

For each of these events, you need to be able to train a proper model (for example one large model or a model for each event). For the CARE2Compare dataset we assumed 1 year of training data with >=70% normal operation is enough to create a normal behavior model.

This notebook shows how to apply the CAREScore to the CARE2Compare dataset ([zenodo](https://doi.org/10.5281/zenodo.10958774)).

To reproduce CARE paper results 'exactly', an additional filter is needed (though only for wind farm C):
- determine cut-in and cut-off wind speeds by power curve analysis
- Remove potentially anomalous data from the training data:
    - Remove rows where the wind speed is outside the normal operation range (below cut-in or above cut-off)
    - Remove rows where the power is zero or near zero (e.g. $P < 0.01$). 

In [1]:
import os
from pathlib import Path

import pandas as pd

from energy_fault_detector.fault_detector import FaultDetector
from energy_fault_detector.config import Config
from energy_fault_detector.evaluation import CAREScore, Care2CompareDataset

2025-11-15 21:20:13.337786: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-15 21:20:13.356397: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-15 21:20:13.356414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-15 21:20:13.356913: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-15 21:20:13.360284: I tensorflow/core/platform/cpu_feature_guar

In [2]:
files_A = [0, 3, 10, 13, 14, 17, 22, 24, 25, 26, 38, 40, 42, 25, 51, 68, 69, 71, 72, 73, 84, 92]

In [5]:
data_dir = Path('../CARE_To_Compare')

In [6]:
c2c = Care2CompareDataset(data_dir)

In [7]:
feature_map = c2c._load_feature_descriptions()["A"].to_dict()

In [8]:
feature_map

{'sensor_name': {0: 'sensor_0',
  1: 'sensor_1',
  2: 'sensor_2',
  3: 'wind_speed_3',
  4: 'wind_speed_4',
  5: 'sensor_5',
  6: 'sensor_6',
  7: 'sensor_7',
  8: 'sensor_8',
  9: 'sensor_9',
  10: 'sensor_10',
  11: 'sensor_11',
  12: 'sensor_12',
  13: 'sensor_13',
  14: 'sensor_14',
  15: 'sensor_15',
  16: 'sensor_16',
  17: 'sensor_17',
  18: 'sensor_18',
  19: 'sensor_19',
  20: 'sensor_20',
  21: 'sensor_21',
  22: 'sensor_22',
  23: 'sensor_23',
  24: 'sensor_24',
  25: 'sensor_25',
  26: 'sensor_26',
  27: 'reactive_power_27',
  28: 'reactive_power_28',
  29: 'power_29',
  30: 'power_30',
  31: 'sensor_31',
  32: 'sensor_32',
  33: 'sensor_33',
  34: 'sensor_34',
  35: 'sensor_35',
  36: 'sensor_36',
  37: 'sensor_37',
  38: 'sensor_38',
  39: 'sensor_39',
  40: 'sensor_40',
  41: 'sensor_41',
  42: 'sensor_42',
  43: 'sensor_43',
  44: 'sensor_44',
  45: 'sensor_45',
  46: 'sensor_46',
  47: 'sensor_47',
  48: 'sensor_48',
  49: 'sensor_49',
  50: 'sensor_50',
  51: 'sensor_

In [9]:
def map_data(data_dict, feature_map, features):
    new_dict = {feature:[] for feature in features}

    sensor2id = {v:k for k, v in feature_map["sensor_name"].items()}
    descriptions = feature_map["description"]
    print("Total datapoints:", len(data_dict["time_stamp"]))
    
    # print(data_dict.keys())
    # print(1/0)
    for k,v in data_dict.items(): #Iterates through each feature and returns only the ones specified in features.
        if k[-3:] == "std":
            continue
        sensor_num = "_".join(k.split("_")[:-1])
        if sensor_num[-1] not in "0123456789":
            continue
        feature = descriptions[sensor2id[sensor_num]]

        if feature in features:
            new_dict[feature] = v

    anomaly_labels = []
    for status in data_dict["status_type_id"]:
        if status==0 or status==2:
            anomaly_labels.append(0)
        else:
            anomaly_labels.append(1)
            
    new_dict["status"] = anomaly_labels
    print("Anomaly datapoints", sum(new_dict["status"]))
    
    new_dict["timestamp"] = data_dict["time_stamp"]
    return new_dict

features = ["Windspeed", "Wind absolute direction", "Wind relative direction", "Ambient temperature", "Grid power", "Rotor rpm", "Generator rpm in latest period", "Pitch angle", "Nacelle direction", "Temperature oil in gearbox", "Temperature in gearbox bearing on high speed shaft", "Temperature in generator bearing 2 (Drive End)", "Temperature in generator bearing 1 (Non-Drive End)", "Nacelle temperature", "Temperature oil in hydraulic group"]

In [10]:
import pandas as pd
import numpy as np

def to_uniform_times(
    df: pd.DataFrame,
    time_col: str = "timestamp",
    rule: str = "10min",
    method: str = "time",   # or "linear", "nearest", "ffill"
    limit: int = 6,  # max consecutive NaNs to fill; use None for no cap
    tz: str = "naive",      # "naive" or a timezone like "UTC"
) -> pd.DataFrame:
    # parse time, handle timezone, set index
    ts = pd.to_datetime(df[time_col], errors="coerce", utc=False)
    if tz and tz != "naive":
        if ts.dt.tz is None:
            ts = ts.dt.tz_localize(tz)
        else:
            ts = ts.dt.tz_convert(tz)
    df = df.drop(columns=[time_col]).copy()
    df.index = ts
    df = df.sort_index()
    df = df[~df.index.isna()]
    df = df[~df.index.duplicated(keep="last")]

    # keep only numeric columns
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_num = df[num_cols].copy()

    # resample to uniform grid
    df_rs = df_num.resample(rule).mean()

    # interpolate
    if method == "ffill":
        df_interp = df_rs.ffill(limit=limit)
    else:
        df_interp = df_rs.interpolate(method=method, limit=limit, limit_direction="both")

    # small safety back/forward fill for edges
    df_interp = df_interp.bfill(limit=limit).ffill(limit=limit)

    # return with timestamp column restored (naive UTC if tz-aware)
    if getattr(df_interp.index, "tz", None) is not None:
        ts_out = df_interp.index.tz_convert("UTC").tz_localize(None)
    else:
        ts_out = df_interp.index
    out = df_interp.reset_index().rename(columns={"index": time_col})
    out[time_col] = ts_out.astype("datetime64[ns]")
    return out

In [46]:
import pandas as pd

REQUIRED_COLS = ["timestamp"]
RECOMMENDED = ["status", "Windspeed", "Wind absolute direction", "Wind relative direction", "Ambient temperature", "Grid power", "Rotor rpm", "Generator rpm in latest period", "Pitch angle", "Nacelle direction", "Temperature oil in gearbox", "Temperature in gearbox bearing on high speed shaft", "Temperature in generator bearing 2 (Drive End)", "Temperature in generator bearing 1 (Non-Drive End)", "Nacelle temperature", "Temperature oil in hydraulic group"]

for i in range(1,5):
    tmp = RECOMMENDED.copy()
    for feature in RECOMMENDED:
        tmp.append(f"{feature}_lvl{i}")
    RECOMMENDED = tmp

def to_training_csv(data_dict, out_path="your_scada.csv"):
    df = pd.DataFrame(data_dict) if "timestamp" in data_dict else (
        pd.DataFrame.from_dict(data_dict, orient="index")
          .sort_index().rename_axis("timestamp").reset_index()
    )
    # keep only the recommended if present
    # print(df.keys())
    # return
    cols = REQUIRED_COLS + [c for c in RECOMMENDED if c in df.columns]
    df = df[cols]
    # sort & clean
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp")
    df.to_csv(out_path, index=False)
    return df

In [ ]:
# i = 0

# for file_id in files_A:
#     print("File id:", file_id)   
#     data = c2c.get_dataset_for_event(file_id, statistics=['average', 'std_dev'])
#     data_dict = map_data(data[0][:10000].to_dict(orient = "list"), feature_map, features)
#     # print(data_dict)
#     if i == len(files_A)-1:
#         to_training_csv(pd.DataFrame(data_dict).drop(columns = ["status"]), f"turbine_data/test/turbine_{1}_nostatus.csv")
#         to_training_csv(pd.DataFrame(data_dict), f"turbine_data/test/turbine_{1}.csv")
#     else:
#         # print(1, data_dict.keys())
#         to_training_csv(pd.DataFrame(data_dict).drop(columns = ["status"]), f"turbine_data/train/turbine_{i+1}_nostatus.csv")
#         # print(2, data_dict.keys())
#         to_training_csv(pd.DataFrame(data_dict), f"turbine_data/train/turbine_{i+1}.csv")
#     i += 1
#     print()

In [12]:
data = c2c.get_dataset_for_event(14, statistics=['average', 'std_dev'])
data_dict = map_data(data[0].head(10000).to_dict(orient = "list"), c2c._load_feature_descriptions()["A"].to_dict(), features)

Total datapoints: 10000
Anomaly datapoints 460


In [ ]:
# from matplotlib import pyplot as plt

# for i in range(10):
#     plt.plot(xs[1000*i:1000*(i+1)], ys[1000*i:1000*(i+1)])
#     plt.show()

In [ ]:
# Use DWT or turn scalogram into curve

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pywt

cols = ["Windspeed", "Wind absolute direction", "Wind relative direction", "Ambient temperature", "Grid power", "Rotor rpm", "Generator rpm in latest period", "Pitch angle", "Nacelle direction", "Temperature oil in gearbox", "Temperature in gearbox bearing on high speed shaft", "Temperature in generator bearing 2 (Drive End)", "Temperature in generator bearing 1 (Non-Drive End)", "Nacelle temperature", "Temperature oil in hydraulic group"]

def load_series(file_path, time_col=None, value_col=None):
    df = pd.read_csv(file_path)

    # Try to find a time-like column if not provided
    candidate_time_cols = [c for c in df.columns if any(k in c.lower() for k in ["time","timestamp","date"])]
    if time_col is None:
        time_col = candidate_time_cols[0] if candidate_time_cols else None

    # Build time array
    if time_col is not None and time_col in df.columns:
        try:
            t_raw = pd.to_datetime(df[time_col])
            t = (t_raw - t_raw.iloc[0]).dt.total_seconds().to_numpy()
        except Exception:
            t = pd.to_numeric(df[time_col], errors="coerce").to_numpy()
            t = t - t[0]
    else:
        t = np.arange(len(df), dtype=float)

    # Pick a numeric value column (fallback: "Windspeed" if present)
    if value_col is None:
        if "Windspeed" in df.columns:
            value_col = "Windspeed"
        else:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != time_col]
            if not numeric_cols:
                for c in df.columns:
                    if c != time_col:
                        coerced = pd.to_numeric(df[c], errors="coerce")
                        if coerced.notna().sum() > len(coerced)//2:
                            numeric_cols.append(c); break
            if not numeric_cols:
                raise ValueError("No numeric signal column found; specify value_col explicitly.")
            value_col = numeric_cols[0]

    x = pd.to_numeric(df[value_col], errors="coerce").to_numpy()

    # Clean and compute dt
    mask = np.isfinite(t) & np.isfinite(x)
    t, x = t[mask], x[mask]
    dt = np.median(np.diff(t)) if len(t) > 1 else 1.0
    if not np.isfinite(dt) or dt <= 0:
        dt = 1.0
    return t, x, dt

def dwt_swt(t, x, dt=None, wavelet="db4", max_level=None, norm=True):
    """
    Stationary Wavelet Transform (undecimated DWT) for a scalogram-like view.
    Returns pseudo-center frequencies for each level and power of the detail coeffs.
    """
    if dt is None or not np.isfinite(dt) or dt <= 0:
        dt = 1.0
    fs = 1.0 / dt

    # Choose a sensible maximum level
    if max_level is None:
        # swt_max_level ensures powers-of-two constraints internally
        max_level = pywt.swt_max_level(len(x))

    # SWT returns list of (approx, detail) for levels 1..max_level
    coeffs = pywt.swt(x, wavelet=wavelet, level=max_level, norm=norm)
    # details[j-1] corresponds to level j detail (same length as x)
    details = [cD for (cA, cD) in coeffs]

    # Power (time-resolved) per level
    # print("details", details)
    power = np.vstack([(d**2) for d in details])

    # Pseudo center frequency for each level j (band ~ [fs/2^(j+1), fs/2^j])
    # Use band mid as a simple representative:
    levels = np.arange(1, max_level+1)
    freqs = fs / (2.0 ** (levels + 0.5))

    # Return with lowest "frequency" first (like your CWT plot)
    # SWT levels increase with j => lower frequency; that's already aligned.
    return freqs, t, details, power

def plot_results(t, x, freqs, power, out_prefix="wavelet"):
    out_dir = Path(".")
    out_dir.mkdir(parents=True, exist_ok=True)

    # Time series
    plt.figure(figsize=(10,4))
    plt.plot(t, x)
    plt.xlabel("Time (s)"); plt.ylabel("Signal"); plt.title("Time Series")
    plt.tight_layout(); plt.savefig(out_dir / f"{out_prefix}_timeseries.png", dpi=150); plt.close()

    # "Scalogram" from SWT detail powers
    # power shape: (levels, time)
    plt.figure(figsize=(10,6))
    extent = [t[0], t[-1], freqs[0], freqs[-1]]
    plt.imshow(power, extent=extent, aspect='auto', origin='lower')
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz) (approx per level)")
    plt.title("Wavelet Power (SWT detail bands)")
    cb = plt.colorbar(); cb.set_label("Power")
    plt.tight_layout(); plt.savefig(out_dir / f"{out_prefix}_scalogram.png", dpi=150); plt.close()

    # "Global" spectrum: average power per level (like integrating over time)
    plt.figure(figsize=(6,6))
    global_spectrum = power.mean(axis=1)
    plt.plot(global_spectrum, freqs)
    plt.xlabel("Average Power"); plt.ylabel("Frequency (Hz) (approx)")
    plt.title("Global Wavelet Spectrum (SWT details)")
    plt.tight_layout(); plt.savefig(out_dir / f"{out_prefix}_global_spectrum.png", dpi=150); plt.close()

    plt.figure(figsize=(10,6))
    for power_lvl in power:
        plt.plot(t, power_lvl)
    plt.title("Power by Level")
    plt.tight_layout(); plt.savefig(out_dir / f"{out_prefix}_powers.png", dpi=150); plt.close()

In [40]:
data_dict = {}
df = pd.read_csv(f'turbine_data/train/turbine_{1}.csv')
data_dict["timestamp"] = np.array(df["timestamp"])
data_dict["status"] = np.array(df["status"])

for col in cols:
    t, x, dt = load_series("turbine_data/train/turbine_1.csv", time_col="timestamp", value_col = col)
    _, _, details, _ = dwt_swt(t, x, dt=dt, max_level=pywt.swt_max_level(len(x)))
    
    for i in range(len(details)):
        data_dict[f"{col}_lvl{i+1}"] = details[i]

In [41]:
data_dict

{'timestamp': array(['2022-08-04 06:10:00', '2022-08-04 06:20:00',
        '2022-08-04 06:30:00', ..., '2022-10-14 01:40:00',
        '2022-10-14 01:50:00', '2022-10-14 02:00:00'], dtype=object),
 'status': array([0, 0, 0, ..., 0, 0, 0]),
 'Windspeed_lvl1': array([ 0.55153757,  0.62932589,  0.62986119, ..., -0.13081253,
         0.17353378,  0.40159474]),
 'Windspeed_lvl2': array([ 0.31262445,  0.55312937,  0.67430923, ..., -0.72588986,
        -0.35705844, -0.00367771]),
 'Windspeed_lvl3': array([ 0.49942862,  0.11471575, -0.01982824, ..., -0.89871157,
        -0.05177828,  0.50920962]),
 'Windspeed_lvl4': array([ 0.34260958, -0.2694479 ,  0.04081805, ..., -1.37963891,
         0.49808372,  0.11332591]),
 'Wind absolute direction_lvl1': array([-33.31162318, -24.27556774, -14.74482108, ..., -49.03069089,
        -47.15215937, -41.5758152 ]),
 'Wind absolute direction_lvl2': array([ -3.50911766,   3.16398717,   3.92095297, ..., -52.95583686,
        -46.45393719, -22.54898193]),
 'Wind 

In [47]:
to_training_csv(data_dict, "turbine_1_wavelets.csv")

,timestamp,status,Windspeed_lvl1,Wind absolute direction_lvl1,Wind relative direction_lvl1,Ambient temperature_lvl1,Grid power_lvl1,Rotor rpm_lvl1,Generator rpm in latest period_lvl1,Pitch angle_lvl1,...,Rotor rpm_lvl4,Generator rpm in latest period_lvl4,Pitch angle_lvl4,Nacelle direction_lvl4,Temperature oil in gearbox_lvl4,Temperature in gearbox bearing on high speed shaft_lvl4,Temperature in generator bearing 2 (Drive End)_lvl4,Temperature in generator bearing 1 (Non-Drive End)_lvl4,Nacelle temperature_lvl4,Temperature oil in hydraulic group_lvl4
0,2022-08-04 06:10:00,0,0.551538,-33.311623,-38.949322,-0.279677,0.022522,0.748628,123.269864,-1.333286,...,-0.099847,-7.521298,2.217481e-01,-1.151543e+00,-0.087928,-0.010741,0.002033,0.064593,-3.246969e-01,8.043430e-02
1,2022-08-04 06:20:00,0,0.629326,-24.275568,-30.697580,-0.393046,0.025981,0.970585,157.210685,-2.305796,...,-0.355896,-39.726799,7.825063e-01,-3.303059e+00,-0.113806,-0.307215,-0.895986,-0.680769,1.650195e-01,2.767077e-01
2,2022-08-04 06:30:00,0,0.629861,-14.744821,-21.323099,-0.483646,0.027820,1.029856,169.625191,-2.821081,...,-0.086925,-10.303347,1.910841e-01,-7.928117e-01,-0.410013,-0.485719,-0.031354,0.021100,1.383034e-01,6.744145e-02
3,2022-08-04 06:40:00,0,0.571272,-5.196192,-10.769927,-0.543585,0.028489,0.944983,160.373205,-2.998741,...,0.000000,-0.981355,7.216450e-16,1.731948e-14,0.103529,0.096035,0.123316,0.123316,-8.937145e-03,1.221245e-15
4,2022-08-04 06:50:00,0,0.489936,2.791069,0.258781,-0.586481,0.028246,0.825944,141.075744,-3.189808,...,0.000000,-5.851618,7.216450e-16,1.731948e-14,-0.039585,0.123316,-0.008937,-0.008937,-3.074530e-02,1.221245e-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2022-10-14 01:20:00,0,-0.811852,-45.587891,-20.692917,0.650247,-0.022034,-2.276197,-273.810221,5.384393,...,0.052604,3.207391,1.129320e-02,1.985652e+00,-0.350065,-0.007493,0.003357,-0.030745,1.165734e-15,-7.493495e-03
9996,2022-10-14 01:30:00,0,-0.475247,-47.756468,-29.959292,0.440347,-0.011041,-1.568741,-185.116781,4.059131,...,1.949800,217.340231,-4.246412e+00,1.425338e+01,1.569644,2.443526,2.769232,1.784425,-3.258034e-01,-1.466115e+00
9997,2022-10-14 01:40:00,0,-0.130813,-49.030691,-39.082930,0.232368,-0.000102,-0.835658,-91.143431,2.645699,...,-3.997615,-435.055478,8.756482e+00,-3.574580e+01,-2.959824,-4.975665,-6.195582,-3.768283,6.851423e-01,3.083140e+00
9998,2022-10-14 01:50:00,0,0.173534,-47.152159,-44.433920,0.033219,0.009660,-0.130557,0.563203,1.120876,...,1.181023,116.849982,-2.609925e+00,1.190335e+01,0.922823,1.210363,1.886772,1.138818,-2.070579e-01,-9.317603e-01


In [ ]:
# python train_hawkeye.py \
#   --csv_path turbine_data/turbine_1.csv \
#   --time_col timestamp \
#   --features "Windspeed" "Wind absolute direction" "Wind relative direction" "Ambient temperature" "Grid power" "Rotor rpm" "Generator rpm in latest period" "Pitch angle" "Nacelle direction" "Temperature oil in gearbox" "Temperature in gearbox bearing on high speed shaft" "Temperature in generator bearing 2 (Drive End)" "Temperature in generator bearing 1 (Non-Drive End)" "Nacelle temperature" "Temperature oil in hydraulic group" \
#   --epochs 15

### Create model per event and evaluate

In [ ]:
# c2c = Care2CompareDataset(data_dir)
# index_column = 'id'  # us time_stamp as index column if you are using the TimestampTransformer
# suffix = ''

# configs = {
#     'A': Config('c2c_configs/windfarm_A.yaml'),
#     'B': Config('c2c_configs/windfarm_B.yaml'),
#     'C': Config('c2c_configs/windfarm_C.yaml'),
# }

# result_files = {
#     'A': f'results_A{suffix}.csv',
#     'B': f'results_B{suffix}.csv',
#     'C': f'results_C{suffix}.csv',
# }

# # whether to ignore the normal index during evaluation
# # For WF A we cannot use the normal index to filter the events, as this would remove complete anomalous events (these are not based on actual SCADA Status codes).
# # For WF B and C this info is however useful, and it would not be interesting to detect anomalies during expected anomalous operation (normal_index=False)
# ignore_normal_idx = {
#     'A': True,
#     'B': False,
#     'C': False,
# }

# for wf in ['A', 'B', 'C']:
#     print('Wind Farm ', wf)
#     # create score object
#     care_score = CAREScore(coverage_beta=0.5, eventwise_f_score_beta=0.5, anomaly_detection_method='criticality')
    
#     # update model config
#     config = configs[wf]
#     c2c.update_c2c_config(config, wf)
    
#     results_file = result_files[wf]
#     if os.path.exists(results_file):
#         care_score.load_evaluated_events(results_file)

#     event_ids = c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == wf, 'event_id']

#     for event_id in event_ids:

#         if not care_score.evaluated_events.empty and event_id in care_score.evaluated_events['event_id'].values:
#             # skip if already evaluated
#             continue
        
#         print(event_id)
#         x_train, x_test = c2c.get_dataset_for_event(event_id=event_id, index_column=index_column)
#         # create normal index
#         y_train = x_train['status_type_id'] == 0
#         y_test = x_test['status_type_id'] == 0
        
#         # drop unnecessary features
#         x_train = x_train.drop(['asset_id', 'status_type_id', 'time_stamp'], axis=1, errors='ignore')
#         x_test = x_test.drop(['asset_id', 'status_type_id', 'time_stamp'], axis=1, errors='ignore')
        
#         # create model
#         model = FaultDetector(config)
#         # train and predict
#         train_results = model.fit(sensor_data=x_train, normal_index=y_train, save_models=False)
#         prediction = model.predict(x_test)
        
#         # evaluate event
#         event_info = c2c.event_info_all[c2c.event_info_all['event_id'] == event_id]
#         event_start = event_info['event_start_id'].iloc[0] if index_column == 'id' else event_info['event_start'].iloc[0]
#         event_end = event_info['event_end_id'].iloc[0] if index_column == 'id' else event_info['event_end'].iloc[0]
#         care_score.evaluate_event(
#             event_id=event_id,
#             event_start=event_start,
#             event_end=event_end,
#             event_label=event_info['event_label'].iloc[0],
#             normal_index=y_test,
#             predicted_anomalies=prediction.predicted_anomalies['anomaly'],
#             ignore_normal_index=ignore_normal_idx[wf]
#         )
        
#         # save evaluated events
#         care_score.save_evaluated_events(results_file)

#     # print final score:
#     print('Final score: ', care_score.get_final_score())

In [ ]:
# # combine results and get final score over all events / wind farms
# all_evaluations = pd.concat([pd.read_csv(f'results_{wf}{suffix}.csv') for wf in ['A', 'B', 'C']])
# all_evaluations.to_csv(f'results_all{suffix}.csv', index=False)
# care_score.load_evaluated_events(f'results_all{suffix}.csv')

# print('Wind farm A:')
# print(
#     care_score.get_final_score(event_selection=c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == 'A', 'event_id'])
# )
# print('Wind farm B:')
# print(
#     care_score.get_final_score(event_selection=c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == 'B', 'event_id'])
# )
# print('Wind farm C:')
# print(
#     care_score.get_final_score(event_selection=c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == 'C', 'event_id'])
# )

# print('overall')
# care_score.get_final_score()

### Create model per asset ID and evaluate each event

Another option is to create a model for asset ID, by combining the training datasets, instead of creating a model for each event separately.
We still evaluate each event separately. Example for WF A:

In [ ]:
# # model config
# wf = 'A'
# config = Config(f'c2c_configs/windfarm_{wf}.yaml')
# c2c.update_c2c_config(config, wf)
# care_score = CAREScore(coverage_beta=0.5, eventwise_f_score_beta=0.5, anomaly_detection_method='criticality')

# for x_train, asset_id, event_ids in c2c.iter_train_datasets_per_asset(wf):
#     x_train = x_train.reset_index(drop=True)

#     # create normal index
#     y_train = x_train['status_type_id'] == 0
#     # drop unnecessary features (keep asset ID as a sort of condition)
#     x_train = x_train.drop(['time_stamp', 'status_type_id'], axis=1)
    
#     # create model
#     model = FaultDetector(config)
#     # train and predict
#     train_results = model.fit(sensor_data=x_train, normal_index=y_train, save_models=False)
    
#     # test and evaluate events for this asset
#     for event_id in event_ids:
#         print(event_id)
#         # test model
#         x_test = c2c.get_dataset_for_event(event_id=event_id, test_only=True)
#         y_test = x_test['status_type_id'] == 0
#         x_test = x_test.drop(['time_stamp', 'asset_id', 'status_type_id'], axis=1)
#         prediction = model.predict(x_test)
        
#         # evaluate event
#         event_info = c2c.event_info_all[c2c.event_info_all['event_id'] == event_id]
#         care_score.evaluate_event(
#             event_id=event_id,
#             event_start_id=event_info['event_start_id'].iloc[0],
#             event_end_id=event_info['event_end_id'].iloc[0],
#             event_label=event_info['event_label'].iloc[0],
#             normal_index=y_test,
#             predicted_anomalies=prediction.predicted_anomalies['anomaly'],
#             ignore_normal_index=True,
#         )
